# TSMC Multi-Stage Topology Cache Inspector
Verify that multi-stage topology extraction is working correctly for TSMC cells

**Purpose**: Check stage-aware topology caches for multi-stage format including:
- `num_stages` (number of stages)
- `stages` list with per-stage transistor/gate info
- `intermediate_gate_widths` with per-gate width sums
- Edge attributes with 5-dim one-hot stage encoding

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

# Set style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10

## 1. Load TSMC Stage-Aware Topology Cache

In [ ]:
# Load TSMC stage-aware cache
stage_aware_cache_path = "/home/tkdgn2907/Deepsets_test/MAML/Projects/data_processing/gnn/topology_cache/stage_aware_topology_cache_tsmc_tsmc_hvt.pth"

print(f"Loading TSMC stage-aware cache from: {stage_aware_cache_path}")
stage_aware_cache = torch.load(stage_aware_cache_path, weights_only=False)

print(f"\nLoaded {len(stage_aware_cache)} cells")
print(f"\nCell names and multi-stage info (first 20):")
for i, cell_name in enumerate(sorted(stage_aware_cache.keys())[:20]):
    cell_data = stage_aware_cache[cell_name]
    output_nodes = cell_data['output_nodes']
    print(f"  [{i:2d}] {cell_name:<40s} Outputs: {output_nodes}")
    
    # Check each output's pull-up and pull-down stages
    for output_name in output_nodes:
        if output_name in cell_data['output_topologies']:
            output_topo = cell_data['output_topologies'][output_name]
            
            # Check pull-up
            pull_up = output_topo['pull_up']
            stage_info_up = pull_up.get('stage_info', {})
            num_stages_up = stage_info_up.get('num_stages', 0)
            
            # Check pull-down
            pull_down = output_topo['pull_down']
            stage_info_down = pull_down.get('stage_info', {})
            num_stages_down = stage_info_down.get('num_stages', 0)
            
            print(f"       {output_name}: Pull-up={num_stages_up}-stage, Pull-down={num_stages_down}-stage")

## 2. Verify Multi-Stage Cache Structure

In [ ]:
# Select a cell to inspect in detail (XOR/XNOR cells are good for multi-stage testing)
# Look for complex cells with more stages
complex_cells = []
for cell_name in stage_aware_cache.keys():
    for keyword in ['XOR', 'XNR', 'AOI', 'OAI', 'MUX']:
        if keyword in cell_name.upper():
            complex_cells.append(cell_name)
            break

if complex_cells:
    cell_name = sorted(complex_cells)[0]
else:
    cell_name = sorted(stage_aware_cache.keys())[0]

cell_cache = stage_aware_cache[cell_name]

print(f"Inspecting TSMC multi-stage cell: {cell_name}")
print("=" * 80)

# Get first output
output_name = cell_cache['output_nodes'][0]
output_topo = cell_cache['output_topologies'][output_name]

print(f"\nOutput: {output_name}")

# Detailed Pull-up inspection
print(f"\n{'='*40}")
print(f"PULL-UP PATH STRUCTURE")
print(f"{'='*40}")
pull_up = output_topo['pull_up']
for key, value in pull_up.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key:25s}: Tensor {list(value.shape)} dtype={value.dtype}")
    elif isinstance(value, dict):
        print(f"  {key:25s}: Dict with keys {list(value.keys())}")
    elif isinstance(value, list):
        if len(value) <= 10:
            print(f"  {key:25s}: {value}")
        else:
            print(f"  {key:25s}: List with {len(value)} items")
    else:
        print(f"  {key:25s}: {type(value).__name__} = {value}")

# Detailed Pull-down inspection
print(f"\n{'='*40}")
print(f"PULL-DOWN PATH STRUCTURE")
print(f"{'='*40}")
pull_down = output_topo['pull_down']
for key, value in pull_down.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key:25s}: Tensor {list(value.shape)} dtype={value.dtype}")
    elif isinstance(value, dict):
        print(f"  {key:25s}: Dict with keys {list(value.keys())}")
    elif isinstance(value, list):
        if len(value) <= 10:
            print(f"  {key:25s}: {value}")
        else:
            print(f"  {key:25s}: List with {len(value)} items")
    else:
        print(f"  {key:25s}: {type(value).__name__} = {value}")

## 3. Inspect Multi-Stage Stage Info Details

In [ ]:
# Detailed stage_info inspection
def print_stage_info(stage_info, path_type):
    print(f"\n{'='*60}")
    print(f"{path_type} STAGE INFO DETAILS")
    print(f"{'='*60}")
    
    # Basic info
    num_stages = stage_info.get('num_stages', 0)
    print(f"\n  Number of stages: {num_stages}")
    
    # Legacy fields for backward compatibility
    if 'stage_type' in stage_info:
        print(f"  Stage type (legacy): {stage_info['stage_type']}")
    
    # Intermediate gates
    intermediate_gates = stage_info.get('intermediate_gates', [])
    print(f"\n  Intermediate gates ({len(intermediate_gates)}):")
    for gate in intermediate_gates:
        print(f"      - {gate}")
    
    # Stages list (multi-stage format)
    stages = stage_info.get('stages', [])
    if stages:
        print(f"\n  Stages list ({len(stages)} stages):")
        for i, stage in enumerate(stages):
            print(f"\n    Stage {i+1}:")
            transistors = stage.get('transistors', [])
            print(f"      Transistors ({len(transistors)}): {transistors}")
            gates = stage.get('gates', [])
            print(f"      Gates ({len(gates)}): {gates}")
    else:
        # Legacy format (stage1_transistors, stage2_transistors)
        print(f"\n  Legacy stage format:")
        for key in stage_info.keys():
            if 'transistors' in key.lower():
                print(f"    {key}: {stage_info[key]}")

# Print for both paths
print_stage_info(pull_up.get('stage_info', {}), 'PULL-UP')
print_stage_info(pull_down.get('stage_info', {}), 'PULL-DOWN')

## 4. Verify Intermediate Gate Widths (Per-Gate)

In [ ]:
# Check intermediate_gate_widths for per-gate width sums
def check_intermediate_gate_widths(stage_info, path_type):
    print(f"\n{'='*60}")
    print(f"{path_type} INTERMEDIATE GATE WIDTHS")
    print(f"{'='*60}")
    
    intermediate_gates = stage_info.get('intermediate_gates', [])
    intermediate_gate_widths = stage_info.get('intermediate_gate_widths', {})
    
    if not intermediate_gates:
        print("  No intermediate gates (single-stage cell)")
        return
    
    print(f"\n  Intermediate gates: {intermediate_gates}")
    print(f"  Width dictionary: {intermediate_gate_widths}")
    
    # Check if widths are per-gate or uniform
    if intermediate_gate_widths:
        widths = list(intermediate_gate_widths.values())
        all_same = len(set(widths)) == 1
        
        print(f"\n  Width values:")
        for gate, width in intermediate_gate_widths.items():
            print(f"      {gate}: {width:.4f} um")
        
        if all_same and len(widths) > 1:
            print(f"\n  WARNING: All gates have same width ({widths[0]:.4f})")
            print(f"           This might indicate incorrect total-width assignment!")
        else:
            print(f"\n  Per-gate widths: VERIFIED (widths differ per gate)")
    else:
        print(f"\n  WARNING: No intermediate_gate_widths found in stage_info!")

# Check for both paths
check_intermediate_gate_widths(pull_up.get('stage_info', {}), 'PULL-UP')
check_intermediate_gate_widths(pull_down.get('stage_info', {}), 'PULL-DOWN')

## 5. Verify Edge Attribute Dimensions (5-dim One-Hot)

In [ ]:
# Check edge_attr dimensions for 5-dimensional one-hot encoding
def check_edge_attr(path_data, path_type):
    print(f"\n{'='*60}")
    print(f"{path_type} EDGE ATTRIBUTE ANALYSIS")
    print(f"{'='*60}")
    
    edge_attr = path_data.get('edge_attr', None)
    
    if edge_attr is None:
        print("  No edge_attr found!")
        return
    
    print(f"  Shape: {list(edge_attr.shape)}")
    print(f"  Expected: [num_edges, 5] for 5-dim one-hot stage encoding")
    
    num_edges, num_features = edge_attr.shape
    
    if num_features == 5:
        print(f"\n  5-dim one-hot encoding: VERIFIED")
        print(f"  Stage encoding: [stage1, stage2, stage3, stage4, stage5+]")
        
        # Count edges per stage
        stage_counts = [0, 0, 0, 0, 0]
        for edge in edge_attr:
            stage_idx = torch.argmax(edge).item()
            stage_counts[stage_idx] += 1
        
        print(f"\n  Edges per stage:")
        for i, count in enumerate(stage_counts):
            if count > 0:
                stage_label = f"Stage {i+1}" if i < 4 else "Stage 5+"
                print(f"      {stage_label}: {count} edges")
    elif num_features == 3:
        print(f"\n  WARNING: 3-dim encoding detected (legacy one_stage/two_stage format)")
        print(f"           Expected 5-dim for multi-stage format!")
    else:
        print(f"\n  UNEXPECTED dimension: {num_features}")

# Check for both paths
check_edge_attr(pull_up, 'PULL-UP')
check_edge_attr(pull_down, 'PULL-DOWN')

## 6. Stage Distribution Across All Cells

In [ ]:
# Analyze stage distribution across all cells
pull_up_stages = []
pull_down_stages = []
cells_with_many_stages = []

for cell_name, cell_data in stage_aware_cache.items():
    for output_name in cell_data['output_nodes']:
        if output_name in cell_data['output_topologies']:
            output_topo = cell_data['output_topologies'][output_name]
            
            # Pull-up
            stage_info_up = output_topo['pull_up'].get('stage_info', {})
            num_up = stage_info_up.get('num_stages', 0)
            pull_up_stages.append(num_up)
            
            # Pull-down
            stage_info_down = output_topo['pull_down'].get('stage_info', {})
            num_down = stage_info_down.get('num_stages', 0)
            pull_down_stages.append(num_down)
            
            # Track cells with 3+ stages
            if num_up >= 3 or num_down >= 3:
                cells_with_many_stages.append((cell_name, output_name, num_up, num_down))

# Print distribution
print("STAGE DISTRIBUTION ACROSS ALL CELLS")
print("=" * 60)

print(f"\nPull-up stage distribution:")
for stage, count in sorted(Counter(pull_up_stages).items()):
    print(f"   {stage}-stage: {count} paths ({count/len(pull_up_stages)*100:.1f}%)")

print(f"\nPull-down stage distribution:")
for stage, count in sorted(Counter(pull_down_stages).items()):
    print(f"   {stage}-stage: {count} paths ({count/len(pull_down_stages)*100:.1f}%)")

print(f"\nCells with 3+ stages (first 15):")
for cell_name, output_name, num_up, num_down in sorted(cells_with_many_stages)[:15]:
    print(f"   {cell_name:<40s} {output_name}: up={num_up}, down={num_down}")

## 7. Visualize Multi-Stage Adjacency Matrices

In [ ]:
# Find a cell with multiple stages for visualization
multi_stage_cell = None
for cell_name, cell_data in stage_aware_cache.items():
    for output_name in cell_data['output_nodes']:
        if output_name in cell_data['output_topologies']:
            output_topo = cell_data['output_topologies'][output_name]
            stage_info_up = output_topo['pull_up'].get('stage_info', {})
            stage_info_down = output_topo['pull_down'].get('stage_info', {})
            
            num_up = stage_info_up.get('num_stages', 0)
            num_down = stage_info_down.get('num_stages', 0)
            
            if num_up >= 3 or num_down >= 3:
                multi_stage_cell = (cell_name, output_name)
                break
    if multi_stage_cell:
        break

if multi_stage_cell:
    cell_name, output_name = multi_stage_cell
else:
    cell_name = sorted(stage_aware_cache.keys())[0]
    output_name = stage_aware_cache[cell_name]['output_nodes'][0]

cell_cache = stage_aware_cache[cell_name]
output_topo = cell_cache['output_topologies'][output_name]
pull_up = output_topo['pull_up']
pull_down = output_topo['pull_down']

print(f"Visualizing: {cell_name} ({output_name})")
print(f"Pull-up: {pull_up['stage_info'].get('num_stages', 0)}-stage")
print(f"Pull-down: {pull_down['stage_info'].get('num_stages', 0)}-stage")

if 'adjacency_matrix' in pull_up and 'adjacency_matrix' in pull_down:
    adj_up = pull_up['adjacency_matrix']
    adj_down = pull_down['adjacency_matrix']
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    # Pull-up
    im1 = axes[0].imshow(adj_up.numpy(), cmap='Blues', aspect='auto', interpolation='nearest')
    num_stages_up = pull_up['stage_info'].get('num_stages', 0)
    axes[0].set_title(f'TSMC Pull-Up Path Adjacency Matrix\n{cell_name} ({output_name})\n'
                     f'{adj_up.shape[0]} nodes, {adj_up.nonzero().shape[0]} edges, {num_stages_up}-stage',
                     fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Destination Node', fontsize=11)
    axes[0].set_ylabel('Source Node', fontsize=11)
    plt.colorbar(im1, ax=axes[0], label='Connection')
    
    if adj_up.shape[0] <= 20:
        axes[0].set_xticks(range(len(pull_up['all_nodes'])))
        axes[0].set_xticklabels(pull_up['all_nodes'], rotation=90, fontsize=8)
        axes[0].set_yticks(range(len(pull_up['all_nodes'])))
        axes[0].set_yticklabels(pull_up['all_nodes'], fontsize=8)
    
    # Pull-down
    im2 = axes[1].imshow(adj_down.numpy(), cmap='Oranges', aspect='auto', interpolation='nearest')
    num_stages_down = pull_down['stage_info'].get('num_stages', 0)
    axes[1].set_title(f'TSMC Pull-Down Path Adjacency Matrix\n{cell_name} ({output_name})\n'
                     f'{adj_down.shape[0]} nodes, {adj_down.nonzero().shape[0]} edges, {num_stages_down}-stage',
                     fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Destination Node', fontsize=11)
    axes[1].set_ylabel('Source Node', fontsize=11)
    plt.colorbar(im2, ax=axes[1], label='Connection')
    
    if adj_down.shape[0] <= 20:
        axes[1].set_xticks(range(len(pull_down['all_nodes'])))
        axes[1].set_xticklabels(pull_down['all_nodes'], rotation=90, fontsize=8)
        axes[1].set_yticks(range(len(pull_down['all_nodes'])))
        axes[1].set_yticklabels(pull_down['all_nodes'], fontsize=8)
    
    plt.tight_layout()
    plt.show()
else:
    print("No adjacency matrices to visualize!")

## 8. Detailed Cell Analysis (XOR/XNOR)

In [ ]:
# Find XOR or XNOR cells for detailed analysis
xor_cells = [name for name in stage_aware_cache.keys() if 'XOR' in name.upper() or 'XNR' in name.upper()]

print(f"Found {len(xor_cells)} XOR/XNOR cells")
print("=" * 80)

for cell_name in sorted(xor_cells)[:5]:  # First 5
    cell_data = stage_aware_cache[cell_name]
    
    print(f"\n{cell_name}")
    print("-" * 60)
    
    for output_name in cell_data['output_nodes']:
        if output_name in cell_data['output_topologies']:
            output_topo = cell_data['output_topologies'][output_name]
            
            # Pull-up details
            pull_up = output_topo['pull_up']
            stage_info_up = pull_up.get('stage_info', {})
            num_up = stage_info_up.get('num_stages', 0)
            stages_up = stage_info_up.get('stages', [])
            intermediate_up = stage_info_up.get('intermediate_gates', [])
            widths_up = stage_info_up.get('intermediate_gate_widths', {})
            
            print(f"  {output_name} Pull-up ({num_up}-stage):")
            print(f"    Nodes: {pull_up['all_nodes']}")
            print(f"    Intermediate gates: {intermediate_up}")
            print(f"    Gate widths: {widths_up}")
            if stages_up:
                for i, stage in enumerate(stages_up):
                    print(f"    Stage {i+1}: trans={stage.get('transistors', [])}, gates={stage.get('gates', [])}")
            
            # Pull-down details
            pull_down = output_topo['pull_down']
            stage_info_down = pull_down.get('stage_info', {})
            num_down = stage_info_down.get('num_stages', 0)
            stages_down = stage_info_down.get('stages', [])
            intermediate_down = stage_info_down.get('intermediate_gates', [])
            widths_down = stage_info_down.get('intermediate_gate_widths', {})
            
            print(f"  {output_name} Pull-down ({num_down}-stage):")
            print(f"    Nodes: {pull_down['all_nodes']}")
            print(f"    Intermediate gates: {intermediate_down}")
            print(f"    Gate widths: {widths_down}")
            if stages_down:
                for i, stage in enumerate(stages_down):
                    print(f"    Stage {i+1}: trans={stage.get('transistors', [])}, gates={stage.get('gates', [])}")

## 9. Per-Gate Width Verification

In [ ]:
# Verify that intermediate gate widths are actually per-gate (not uniform)
print("PER-GATE WIDTH VERIFICATION")
print("=" * 80)

uniform_width_cells = []
per_gate_width_cells = []
no_intermediate_cells = []

for cell_name, cell_data in stage_aware_cache.items():
    for output_name in cell_data['output_nodes']:
        if output_name in cell_data['output_topologies']:
            output_topo = cell_data['output_topologies'][output_name]
            
            for path_type in ['pull_up', 'pull_down']:
                path_data = output_topo[path_type]
                stage_info = path_data.get('stage_info', {})
                intermediate_gates = stage_info.get('intermediate_gates', [])
                widths = stage_info.get('intermediate_gate_widths', {})
                
                if not intermediate_gates:
                    no_intermediate_cells.append((cell_name, output_name, path_type))
                elif len(widths) > 1:
                    width_values = list(widths.values())
                    if len(set(width_values)) == 1:
                        uniform_width_cells.append((cell_name, output_name, path_type, width_values[0]))
                    else:
                        per_gate_width_cells.append((cell_name, output_name, path_type, widths))

print(f"\nSummary:")
print(f"   Cells with no intermediate gates: {len(no_intermediate_cells)}")
print(f"   Cells with uniform width (potential issue): {len(uniform_width_cells)}")
print(f"   Cells with per-gate widths (correct): {len(per_gate_width_cells)}")

if uniform_width_cells:
    print(f"\nWARNING: Cells with uniform width (first 10):")
    for cell_name, output_name, path_type, width in uniform_width_cells[:10]:
        print(f"   {cell_name} {output_name} {path_type}: width={width:.4f}")

if per_gate_width_cells:
    print(f"\nCells with per-gate widths (first 5):")
    for cell_name, output_name, path_type, widths in per_gate_width_cells[:5]:
        print(f"   {cell_name} {output_name} {path_type}:")
        for gate, w in widths.items():
            print(f"      {gate}: {w:.4f} um")

## 10. Summary Statistics

In [ ]:
print("=" * 100)
print("TSMC MULTI-STAGE CACHE VERIFICATION SUMMARY")
print("=" * 100)

# Count cache format types
has_num_stages = 0
has_stages_list = 0
has_intermediate_widths = 0
has_5dim_edge_attr = 0
has_3dim_edge_attr = 0
total_paths = 0

for cell_data in stage_aware_cache.values():
    for output_topo in cell_data['output_topologies'].values():
        for path_type in ['pull_up', 'pull_down']:
            path_data = output_topo[path_type]
            stage_info = path_data.get('stage_info', {})
            total_paths += 1
            
            if 'num_stages' in stage_info:
                has_num_stages += 1
            if 'stages' in stage_info:
                has_stages_list += 1
            if 'intermediate_gate_widths' in stage_info:
                has_intermediate_widths += 1
            
            edge_attr = path_data.get('edge_attr', None)
            if edge_attr is not None:
                if edge_attr.shape[1] == 5:
                    has_5dim_edge_attr += 1
                elif edge_attr.shape[1] == 3:
                    has_3dim_edge_attr += 1

print(f"\nTotal cells: {len(stage_aware_cache)}")
print(f"Total paths: {total_paths}")

print(f"\nMulti-Stage Format Support:")
print(f"   Paths with num_stages: {has_num_stages}/{total_paths} ({has_num_stages/total_paths*100:.1f}%)")
print(f"   Paths with stages list: {has_stages_list}/{total_paths} ({has_stages_list/total_paths*100:.1f}%)")
print(f"   Paths with intermediate_gate_widths: {has_intermediate_widths}/{total_paths} ({has_intermediate_widths/total_paths*100:.1f}%)")

print(f"\nEdge Attribute Dimensions:")
print(f"   5-dim (multi-stage): {has_5dim_edge_attr}/{total_paths} ({has_5dim_edge_attr/total_paths*100:.1f}%)")
print(f"   3-dim (legacy): {has_3dim_edge_attr}/{total_paths} ({has_3dim_edge_attr/total_paths*100:.1f}%)")

print(f"\nStage Distribution:")
print(f"   Pull-up: {dict(Counter(pull_up_stages))}")
print(f"   Pull-down: {dict(Counter(pull_down_stages))}")

print(f"\n{'=' * 100}")
if has_num_stages == total_paths and has_5dim_edge_attr == total_paths:
    print("VERIFICATION PASSED: Multi-stage format is properly applied!")
elif has_num_stages > 0:
    print("PARTIAL: Some paths have multi-stage format, some may have legacy format.")
else:
    print("LEGACY FORMAT: Cache still uses old one_stage/two_stage format.")
print("=" * 100)